# Spark Fundamentals — Data Cleaning, Transformation & Aggregation

**Objective:** Understand Spark fundamentals and perform data cleaning, transformation, and aggregation using DataFrames.

## Step 1: MapReduce vs Spark — quick notes

- **MapReduce** writes intermediate results to disk between the map and reduce phases. That's fine for one-pass batch jobs but it gets slow when a pipeline needs multiple passes over the same data (e.g. iterative filtering + aggregation like below), because every stage means another round trip to disk.
- **Spark** keeps data in memory (as RDDs / DataFrames) across stages using its DAG execution engine, so chained transformations don't hit disk until an action forces it. That's the main reason it's commonly benchmarked as much faster than MapReduce for iterative/interactive workloads.
- Spark DataFrames also give a higher-level, SQL-like API on top of that engine (`filter`, `groupBy`, `agg`, etc.) which is what we use below instead of writing raw map/reduce functions.


## Step 2: Start Spark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, StringType

spark = SparkSession.builder \
    .appName("RetailDataCleaningAndAggregation") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8")  \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
spark


## Step 3: Load Data

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Spark Basics") \
    .getOrCreate()

df = spark.read.csv("Superstore_Dataset.csv",header=True,inferSchema=True)

df.show(5)
df.printSchema()

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [4]:
df.show(5, truncate=25)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+-------------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|             Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+-------------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases

## Step 3b: DataFrame immutability

Every transformation below (`filter`, `withColumn`, `drop`, ...) returns a **new** DataFrame — the original `df` is never modified in place. That's why you'll see the pattern `df_clean = df.dropDuplicates()` throughout instead of something like `df.dropDuplicates(inplace=True)` (which doesn't even exist in Spark, unlike pandas). Transformations are also **lazy**: Spark just builds up a logical plan (a DAG) and only actually runs it when an **action** like `.show()`, `.count()`, or `.collect()` is called.


## Step 4: Data Cleaning

### 4a. Check for nulls / missing values per column

In [5]:
df.select([
    F.count(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), c)).alias(c)
    for c in df.columns
]).show(truncate=False)


+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|0     |0       |0         |0        |0        |0          |0            |0      |0      |0   |0    |0          |0     |0         |0       |0           |0           |0    |0       |0       |0     |
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



### 4b. Remove exact duplicate rows

In [6]:
before = df.count()
df_dedup = df.dropDuplicates()
after = df_dedup.count()
print(f"Rows before dedup: {before}, after dedup: {after}, duplicates removed: {before - after}")


Rows before dedup: 9997, after dedup: 9994, duplicates removed: 3


### 4c. Handle nulls

- `Age` came in as a string column because of a few `'NA'` values mixed with numbers — a classic messy-schema issue. Cast it to numeric first, coercion turns the bad strings into nulls, then fill.
- Rows missing `Total_Amount` or `Quantity` (the numbers that actually drive the aggregation) are dropped since there's nothing safe to fill them with.
- `Payment_Method` is genuinely optional in the source data (guest checkouts etc.), so those nulls are filled with `'Unknown'` instead of dropping the row.
- Blank strings in `City` are normalized to `null` first so `dropna`/`fillna` treat them consistently.


In [7]:
df_clean = df_dedup \
    .withColumn("City", F.when(F.trim(F.col("City")) == "", None).otherwise(F.col("City"))) \
    .withColumn("Age", F.expr("try_cast(try_cast(Age AS DOUBLE) AS INT)"))

# fill categorical/optional nulls
df_clean = df_clean.fillna({"Payment_Method": "Unknown", "City": "Unknown"})

# drop rows where a core numeric measure is missing - can't safely impute these
df_clean = df_clean.dropna(subset=["Quantity", "Total_Amount", "Unit_Price"])

# fill remaining Age nulls with the column median (robust to outliers, better than mean here)
age_median = df_clean.approxQuantile("Age", [0.5], 0.01)[0]
df_clean = df_clean.fillna({"Age": int(age_median)})

print("Rows after full cleaning:", df_clean.count())
df_clean.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_clean.columns
]).show(truncate=False)


{"ts": "2026-07-21 22:47:50.681", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `Age` cannot be resolved. Did you mean one of the following? [`City`, `Sales`, `State`, `Region`, `Segment`]. SQLSTATE: 42703", "context": {"errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o422.withColumn.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `Age` cannot be resolved. Did you mean one of the following? [`City`, `Sales`, `State`, `Region`, `Segment`]. SQLSTATE: 42703; line 1 pos 18;\n'Project [Row ID#141, Order ID#142, Order Date#143, Ship Date#144, Ship Mode#145, Customer ID#146, Customer Name#147, Segment#148, Country#149, City#615, State#151, Postal Code#152, Region#153, Product ID#154, Category#155, Sub-Category#156, Product Name

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `Age` cannot be resolved. Did you mean one of the following? [`City`, `Sales`, `State`, `Region`, `Segment`]. SQLSTATE: 42703; line 1 pos 18;
'Project [Row ID#141, Order ID#142, Order Date#143, Ship Date#144, Ship Mode#145, Customer ID#146, Customer Name#147, Segment#148, Country#149, City#615, State#151, Postal Code#152, Region#153, Product ID#154, Category#155, Sub-Category#156, Product Name#157, Sales#158, Quantity#159, Discount#160, Profit#161, try_cast(try_cast('Age as double) as int) AS Age#616]
+- Project [Row ID#141, Order ID#142, Order Date#143, Ship Date#144, Ship Mode#145, Customer ID#146, Customer Name#147, Segment#148, Country#149, CASE WHEN (trim(City#150, None) = ) THEN cast(null as string) ELSE City#150 END AS City#615, State#151, Postal Code#152, Region#153, Product ID#154, Category#155, Sub-Category#156, Product Name#157, Sales#158, Quantity#159, Discount#160, Pr

### 4d. Standardize inconsistent categorical values

`Product_Category` had mixed casing and a couple of alternate spellings (`Electronic` vs `Electronics`, `Cloths` vs `Clothing`, etc.) — exactly the kind of thing that silently breaks a `groupBy` if you don't catch it.

In [ ]:
df_clean.groupBy("Product_Category").count().orderBy(F.desc("count")).show(20, truncate=False)


+----------------+-----+
|Product_Category|count|
+----------------+-----+
|Book            |1104 |
|furniture       |1088 |
|Furniture       |1081 |
|FURNITURE       |1080 |
|Grocery         |1078 |
|sports          |1075 |
|grocery         |1066 |
|BEAUTY          |1066 |
|books           |1059 |
|beauty          |1056 |
|Beauty          |1056 |
|Sport           |1054 |
|Groceries       |1037 |
|Sports          |1034 |
|Books           |1032 |
|Clothing        |853  |
|electronics     |835  |
|Cloths          |828  |
|ELECTRONICS     |824  |
|Electronics     |804  |
+----------------+-----+
only showing top 20 rows


In [ ]:
category_map = {
    "electronics": "Electronics", "electronic": "Electronics",
    "clothing": "Clothing", "cloths": "Clothing",
    "grocery": "Grocery", "groceries": "Grocery",
    "furniture": "Furniture",
    "books": "Books", "book": "Books",
    "beauty": "Beauty",
    "sports": "Sports", "sport": "Sports",
}
map_expr = F.create_map([F.lit(x) for pair in category_map.items() for x in pair])

df_clean = df_clean.withColumn(
    "Product_Category",
    F.coalesce(map_expr[F.lower(F.trim(F.col("Product_Category")))], F.col("Product_Category"))
)

df_clean.groupBy("Product_Category").count().orderBy(F.desc("count")).show(truncate=False)


+----------------+-----+
|Product_Category|count|
+----------------+-----+
|Electronics     |3250 |
|Furniture       |3249 |
|Clothing        |3248 |
|Books           |3195 |
|Grocery         |3181 |
|Beauty          |3178 |
|Sports          |3163 |
+----------------+-----+



## Step 5: Filter Data

### 5a. Filter by age range (e.g. 25–40)

In [ ]:
df_clean.filter((F.col("Age") >= 25) & (F.col("Age") <= 40)) \
    .select("Transaction_ID", "Customer_ID", "Age", "Product_Category", "Total_Amount") \
    .show(5)


+--------------+-----------+---+----------------+------------+
|Transaction_ID|Customer_ID|Age|Product_Category|Total_Amount|
+--------------+-----------+---+----------------+------------+
|       T104882|     C05337| 36|       Furniture|     1955.78|
|       T105843|     C04142| 37|       Furniture|     2331.32|
|       T111252|     C01605| 39|     Electronics|      1471.9|
|       T117567|     C00277| 29|          Sports|      358.94|
|       T108236|     C06964| 28|       Furniture|     1939.68|
+--------------+-----------+---+----------------+------------+
only showing top 5 rows


### 5b. Filter by category

In [ ]:
df_clean.filter(F.col("Product_Category") == "Electronics") \
    .select("Transaction_ID", "Product_Category", "Quantity", "Total_Amount") \
    .show(5)


+--------------+----------------+--------+------------+
|Transaction_ID|Product_Category|Quantity|Total_Amount|
+--------------+----------------+--------+------------+
|       T107838|     Electronics|     5.0|      2314.8|
|       T111252|     Electronics|     9.0|      1471.9|
|       T103405|     Electronics|     7.0|     2955.24|
|       T106050|     Electronics|    10.0|      848.24|
|       T102840|     Electronics|    10.0|     1703.62|
+--------------+----------------+--------+------------+
only showing top 5 rows


### 5c. Filter by region

In [ ]:
df_clean.filter(F.col("Region") == "West") \
    .select("Transaction_ID", "Region", "Country", "Total_Amount") \
    .show(5)


+--------------+------+-------+------------+
|Transaction_ID|Region|Country|Total_Amount|
+--------------+------+-------+------------+
|       T107838|  West|     UK|      2314.8|
|       T102523|  West|    USA|      674.15|
|       T116544|  West|     UK|      763.97|
|       T106709|  West|     UK|     1756.34|
|       T111564|  West|     UK|      149.08|
+--------------+------+-------+------------+
only showing top 5 rows


### 5d. Combine multiple filter conditions (age + category + order status)

In [ ]:
df_clean.filter(
    (F.col("Age").between(25, 40)) &
    (F.col("Product_Category") == "Electronics") &
    (F.col("Order_Status") == "Completed")
).show(5)


+--------------+-----------+----------------+--------------------+---+------+-------+-------+--------------------+----------------+--------+----------+------------+----------+--------------+------------+
|Transaction_ID|Customer_ID|            Name|               Email|Age|Gender|Country| Region|                City|Product_Category|Quantity|Unit_Price|Total_Amount|Order_Date|Payment_Method|Order_Status|
+--------------+-----------+----------------+--------------------+---+------+-------+-------+--------------------+----------------+--------+----------+------------+----------+--------------+------------+
|       T100092|     C05385|Teresa Rodriguez|thomasharvey@exam...| 33| Other| France|Central|            Rosebury|     Electronics|     6.0|      94.8|      528.22|2025-09-18|    Debit Card|   Completed|
|       T110680|     C04554|Michael Erickson|samantha29@exampl...| 28| Other|    USA|  North|           Port Anne|     Electronics|    10.0|    318.79|     3319.05|2025-03-30|         

## Step 6: Transform Data — rename columns & cast types

In [ ]:
df_transformed = df_clean \
    .withColumnRenamed("Total_Amount", "Order_Total") \
    .withColumnRenamed("Product_Category", "Category") \
    .withColumn("Quantity", F.col("Quantity").cast(IntegerType())) \
    .withColumn("Order_Total", F.col("Order_Total").cast(DoubleType())) \
    .withColumn("Order_Date", F.to_date(F.col("Order_Date"), "yyyy-MM-dd"))

df_transformed.printSchema()
df_transformed.show(5)


root
 |-- Transaction_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Age: integer (nullable = false)
 |-- Gender: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- City: string (nullable = false)
 |-- Category: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Unit_Price: double (nullable = true)
 |-- Order_Total: double (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Payment_Method: string (nullable = false)
 |-- Order_Status: string (nullable = true)



+--------------+-----------+------------------+--------------------+---+------+-------+------+----------------+-----------+--------+----------+-----------+----------+--------------+------------+
|Transaction_ID|Customer_ID|              Name|               Email|Age|Gender|Country|Region|            City|   Category|Quantity|Unit_Price|Order_Total|Order_Date|Payment_Method|Order_Status|
+--------------+-----------+------------------+--------------------+---+------+-------+------+----------------+-----------+--------+----------+-----------+----------+--------------+------------+
|       T107838|     C04160|      Susan Miller|christineoliver@e...| 71| Other|     UK|  West|     Sanchezbury|Electronics|       5|    468.54|     2314.8|2025-03-28|          Cash|     Pending|
|       T104882|     C05337|      Kara Johnson|timothyferguson@e...| 36| Other|    USA| North|Port Michaelfurt|  Furniture|       8|    264.01|    1955.78|2024-07-21|          Cash|   Completed|
|       T107229|     C024

## Step 7: Aggregation — basic calculations

In [ ]:
df_transformed.select(
    F.count("*").alias("total_rows"),
    F.round(F.avg("Order_Total"), 2).alias("avg_order_value"),
    F.round(F.sum("Order_Total"), 2).alias("total_revenue"),
    F.min("Order_Total").alias("min_order_value"),
    F.max("Order_Total").alias("max_order_value"),
).show()


+----------+---------------+-------------+---------------+---------------+
|total_rows|avg_order_value|total_revenue|min_order_value|max_order_value|
+----------+---------------+-------------+---------------+---------------+
|     22464|        1396.08| 3.13615853E7|            5.5|        5451.97|
+----------+---------------+-------------+---------------+---------------+



## Step 8: Group Data

### 8a. Revenue and order count by category

In [ ]:
category_summary = df_transformed.groupBy("Category").agg(
    F.count("*").alias("order_count"),
    F.round(F.sum("Order_Total"), 2).alias("total_revenue"),
    F.round(F.avg("Order_Total"), 2).alias("avg_order_value"),
).orderBy(F.desc("total_revenue"))

category_summary.show(truncate=False)


+-----------+-----------+-------------+---------------+
|Category   |order_count|total_revenue|avg_order_value|
+-----------+-----------+-------------+---------------+
|Furniture  |3249       |4573560.81   |1407.68        |
|Clothing   |3248       |4492811.05   |1383.25        |
|Sports     |3163       |4475577.09   |1414.98        |
|Beauty     |3178       |4469509.26   |1406.39        |
|Books      |3195       |4461511.59   |1396.4         |
|Grocery    |3181       |4452406.72   |1399.69        |
|Electronics|3250       |4436208.78   |1364.99        |
+-----------+-----------+-------------+---------------+



### 8b. Revenue by region, with a condition on the aggregated result (HAVING-style filter)

Only keep regions whose average order value is above the overall average — this is the "apply conditions on aggregated results" part: you can't filter on an aggregate with a plain `.filter()` before the `groupBy`, it has to come after `.agg()`.

In [ ]:
overall_avg = df_transformed.agg(F.avg("Order_Total")).collect()[0][0]

region_summary = df_transformed.groupBy("Region").agg(
    F.count("*").alias("order_count"),
    F.round(F.sum("Order_Total"), 2).alias("total_revenue"),
    F.round(F.avg("Order_Total"), 2).alias("avg_order_value"),
).filter(F.col("avg_order_value") > overall_avg) \
 .orderBy(F.desc("avg_order_value"))

print(f"Overall average order value: {overall_avg:.2f}")
region_summary.show(truncate=False)


Overall average order value: 1396.08


+-------+-----------+-------------+---------------+
|Region |order_count|total_revenue|avg_order_value|
+-------+-----------+-------------+---------------+
|Central|4519       |6392052.4    |1414.48        |
|East   |4526       |6377510.91   |1409.08        |
|South  |4482       |6303980.47   |1406.51        |
+-------+-----------+-------------+---------------+



### 8c. Multi-level grouping — category within region

In [ ]:
region_category_summary = df_transformed.groupBy("Region", "Category").agg(
    F.count("*").alias("order_count"),
    F.round(F.sum("Order_Total"), 2).alias("total_revenue"),
).orderBy("Region", F.desc("total_revenue"))

region_category_summary.show(20, truncate=False)


+-------+-----------+-----------+-------------+
|Region |Category   |order_count|total_revenue|
+-------+-----------+-----------+-------------+
|Central|Furniture  |677        |960256.4     |
|Central|Sports     |645        |950069.73    |
|Central|Beauty     |644        |941995.24    |
|Central|Books      |644        |911161.8     |
|Central|Electronics|636        |895513.9     |
|Central|Grocery    |636        |884340.16    |
|Central|Clothing   |637        |848715.17    |
|East   |Furniture  |674        |997294.87    |
|East   |Clothing   |663        |940948.46    |
|East   |Books      |656        |929056.99    |
|East   |Electronics|675        |923933.19    |
|East   |Grocery    |623        |876434.32    |
|East   |Sports     |626        |871872.78    |
|East   |Beauty     |609        |837970.3     |
|North  |Clothing   |695        |969217.81    |
|North  |Beauty     |657        |913789.42    |
|North  |Electronics|669        |912360.86    |
|North  |Grocery    |635        |895245.

## Step 9: Wide transformations & shuffles (basic idea)

- A **narrow transformation** (`filter`, `withColumn`, `select`) only needs the data already sitting in its own partition — no data has to move between executors.
- A **wide transformation** (`groupBy`, `join`, `orderBy`, `distinct`) needs rows that share a key to end up on the same partition, so Spark has to physically move data across the cluster to make that happen. That data movement step is the **shuffle**.
- Every `groupBy().agg()` call in Step 8 above triggers a shuffle. You can see this in the Spark UI (or `.explain()` below) as an `Exchange` step in the physical plan — that's the shuffle boundary.
- Shuffles are the main thing that slows Spark jobs down (disk + network I/O + serialization), so in a real pipeline you'd want to minimize how many wide transformations you chain, and pick sensible partitioning where possible.


In [ ]:
category_summary.explain()


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_revenue#961 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_revenue#961 DESC NULLS LAST, 8), ENSURE_REQUIREMENTS, [plan_id=1611]
      +- HashAggregate(keys=[Category#822], functions=[count(1), sum(Order_Total#824), avg(Order_Total#824)])
         +- Exchange hashpartitioning(Category#822, 8), ENSURE_REQUIREMENTS, [plan_id=1608]
            +- HashAggregate(keys=[Category#822], functions=[partial_count(1), partial_sum(Order_Total#824), partial_avg(Order_Total#824)])
               +- HashAggregate(keys=[Customer_ID#18, Quantity#27, Unit_Price#28, Name#19, Country#23, Region#24, Email#20, Payment_Method#31, Total_Amount#29, Age#21, City#25, Transaction_ID#17, Product_Category#26, Gender#22, Order_Date#30, Order_Status#32], functions=[])
                  +- Exchange hashpartitioning(Customer_ID#18, Quantity#27, Unit_Price#28, Name#19, Country#23, Region#24, Email#20, Payment_Method#31, Total_Amount#2

## Step 10: Full pipeline — load → clean → filter → transform → aggregate

In [ ]:
def build_retail_pipeline(path: str):
    """End-to-end pipeline: takes a raw CSV path, returns a clean summary DataFrame."""

    # 1. Load
    raw = spark.read.csv(path, header=True, inferSchema=True)

    # 2. Clean
    clean = raw.dropDuplicates()
    clean = clean.withColumn("City", F.when(F.trim(F.col("City")) == "", None).otherwise(F.col("City")))
    clean = clean.withColumn("Age", F.expr("try_cast(try_cast(Age AS DOUBLE) AS INT)"))
    category_map = {
        "electronics": "Electronics", "electronic": "Electronics",
        "clothing": "Clothing", "cloths": "Clothing",
        "grocery": "Grocery", "groceries": "Grocery",
        "furniture": "Furniture",
        "books": "Books", "book": "Books",
        "beauty": "Beauty",
        "sports": "Sports", "sport": "Sports",
    }
    map_expr = F.create_map([F.lit(x) for pair in category_map.items() for x in pair])

    clean = clean.fillna({"Payment_Method": "Unknown", "City": "Unknown"})
    clean = clean.dropna(subset=["Quantity", "Total_Amount", "Unit_Price"])
    med_age = clean.approxQuantile("Age", [0.5], 0.01)[0]
    clean = clean.fillna({"Age": int(med_age)})
    clean = clean.withColumn(
        "Product_Category",
        F.coalesce(map_expr[F.lower(F.trim(F.col("Product_Category")))], F.col("Product_Category"))
    )

    # 3. Filter - keep completed orders from adults only, as an example business rule
    filtered = clean.filter((F.col("Order_Status") == "Completed") & (F.col("Age") >= 18))

    # 4. Transform
    transformed = filtered \
        .withColumnRenamed("Total_Amount", "Order_Total") \
        .withColumnRenamed("Product_Category", "Category") \
        .withColumn("Order_Total", F.col("Order_Total").cast(DoubleType()))

    # 5. Aggregate
    summary = transformed.groupBy("Region", "Category").agg(
        F.count("*").alias("order_count"),
        F.round(F.sum("Order_Total"), 2).alias("total_revenue"),
        F.round(F.avg("Order_Total"), 2).alias("avg_order_value"),
    ).orderBy("Region", F.desc("total_revenue"))

    return summary


pipeline_result = build_retail_pipeline("../data/dataset.csv")
pipeline_result.show(30, truncate=False)


+-------+-----------+-----------+-------------+---------------+
|Region |Category   |order_count|total_revenue|avg_order_value|
+-------+-----------+-----------+-------------+---------------+
|Central|Beauty     |181        |263041.76    |1453.27        |
|Central|Furniture  |174        |232343.76    |1335.31        |
|Central|Electronics|164        |229159.33    |1397.31        |
|Central|Clothing   |171        |221076.52    |1292.85        |
|Central|Grocery    |150        |211748.54    |1411.66        |
|Central|Sports     |153        |202523.83    |1323.69        |
|Central|Books      |135        |188347.55    |1395.17        |
|East   |Books      |171        |244661.67    |1430.77        |
|East   |Furniture  |172        |231969.39    |1348.66        |
|East   |Beauty     |168        |231943.04    |1380.61        |
|East   |Grocery    |158        |231113.32    |1462.74        |
|East   |Clothing   |154        |223421.9     |1450.79        |
|East   |Sports     |157        |215564.

### Save pipeline output

In [ ]:
pipeline_result.toPandas().to_csv("../output/results.csv", index=False)
print("Saved to ../output/results.csv")


/usr/local/lib/python3.12/dist-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/dist-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


Saved to ../output/results.csv


In [ ]:
spark.stop()